# MDR2DATA.ipynb

Original Bi-CoT research notebook, recovered from the author’s AI_Study archive. Outputs, execution state, and machine metadata were removed; API credentials now come from `OPENAI_API_KEY`, and local paths now use `../data/workspace` relative to this notebook directory. See `docs/WORKFLOW.md` before selecting cells. Cells include experimental alternatives and data writes; this is not a single Run All pipeline.


In [ ]:
import os
import json
from collections import defaultdict

# 경로 세팅
json_path = "../data/workspace/q03_without_12_beam2_topk2.json"
BASE_PATH = "../data/workspace/CoT_123"  # 원하는 베이스 폴더로 변경하세요

# 아이디별로 엔트리 모으기
id2entries = defaultdict(list)

with open(json_path, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        entry = json.loads(line)
        _id = entry["_id"]
        id2entries[_id].append(entry)

total_ids = len(id2entries)
print(f"총 {total_ids}개 아이디의 데이터를 처리합니다.")

for idx, (_id, entries) in enumerate(id2entries.items(), 1):  # enumerate 시작값을 1로
    if len(entries) != 2:
        print(f"⚠️ {_id}는 엔트리가 {len(entries)}개입니다. (2개 아님, 건너뜀)")
        continue

    save_paths = [
        #os.path.join(BASE_PATH, _id, "0", "q.json"),
        os.path.join(BASE_PATH, _id, "3", "0", "q.json"),
        os.path.join(BASE_PATH, _id, "3", "1", "q.json"),
    ]
    for i, entry in enumerate(entries):
        save_path = save_paths[i]
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        with open(save_path, "w", encoding="utf-8") as f:
            json.dump(entry, f, ensure_ascii=False, indent=2)
    print(f"[{idx}/{total_ids}] {_id}: 3/0/q.json, 3/1/q.json 저장 완료")

print("✅ 모든 아이디 처리 및 저장 완료")

In [ ]:
import os
import json

ID2ANS_PATH = "../data/workspace/id2answer_self_contained.json"
BASE_DIR = "../data/workspace/Last_CoT_123"

def safe_load_json(path):
    try:
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    except Exception as e:
        print(f"[WARN] JSON load failed: {path} ({e})")
        return None

# 1) id 목록 로드
with open(ID2ANS_PATH, "r", encoding="utf-8") as f:
    id2answer = json.load(f)

ids = sorted(id2answer.keys())

# 2) 각 id의 3/cot2/a.json에서 question 출력
results = {}  # {id: question}
missing = []
no_question = []

for ex_id in ids:
    a_path = os.path.join(BASE_DIR, ex_id, "3", "cot2", "a.json")
    if not os.path.exists(a_path):
        missing.append(ex_id)
        continue

    data = safe_load_json(a_path)
    if not data:
        missing.append(ex_id)
        continue

    q = data.get("question", None)
    if q is None:
        no_question.append(ex_id)
        continue

    results[ex_id] = str(q).strip()

# 3) 출력
print("=== Questions (id -> question) ===")
for k in sorted(results.keys()):
    print(f"{k}: {results[k]}")

print("\n=== Summary ===")
print(f"Total ids in id2answer_self_contained.json: {len(ids)}")
print(f"Found questions: {len(results)}")
print(f"Missing files or failed parse: {len(missing)}")
print(f"No 'question' field: {len(no_question)}")

# (옵션) 결과를 파일로 저장하고 싶다면 주석 해제
# out_path = "../data/workspace/id2question_from_cot2.json"
# with open(out_path, "w", encoding="utf-8") as f:
#     json.dump(results, f, ensure_ascii=False, indent=2)
# print(f"\nSaved to: {out_path}")

In [ ]:
import os
import json

ID2ANS_PATH = "../data/workspace/id2answer_self_contained.json"
BASE_DIR    = "../data/workspace/Last_CoT_123"

def safe_load_json(path):
    try:
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    except Exception as e:
        print(f"[WARN] JSON load failed: {path} ({e})")
        return None

def safe_write_json(path, data):
    try:
        with open(path, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2)
        return True
    except Exception as e:
        print(f"[WARN] JSON write failed: {path} ({e})")
        return False

def main():
    # 1) id -> answer 로드
    with open(ID2ANS_PATH, "r", encoding="utf-8") as f:
        id2answer = json.load(f)

    ids = list(id2answer.keys())
    total = len(ids)

    success = 0
    missing = 0
    load_fail = 0
    write_fail = 0

    print(f"Total IDs to process: {total}")

    # 2) 각 id의 3/cot2/a.json에서 answer/evidence_title 업데이트
    for _id in ids:
        a_path = os.path.join(BASE_DIR, _id, "3", "cot2", "a.json")
        if not os.path.exists(a_path):
            print(f"[SKIP] Missing file: {a_path}")
            missing += 1
            continue

        data = safe_load_json(a_path)
        if data is None:
            load_fail += 1
            continue

        # 업데이트: answer, evidence_title
        new_answer = str(id2answer[_id]).strip()
        data["answer"] = new_answer
        data["evidence_title"] = "No evidence"

        if safe_write_json(a_path, data):
            success += 1
            print(f"[{success}/{total}] SUCCESS: {_id} -> answer set to '{new_answer}', evidence_title='No evidence'")
        else:
            write_fail += 1

    # 3) 요약
    print("\n=== Summary ===")
    print(f"Total IDs           : {total}")
    print(f"Updated successfully: {success}")
    print(f"Missing files       : {missing}")
    print(f"JSON load failed    : {load_fail}")
    print(f"JSON write failed   : {write_fail}")

if __name__ == "__main__":
    main()

In [ ]:
import os, json

def find_answer_recursive(obj):
    """dict/list 어디에 있어도 'answer' 키(대소문자 무관)에서 문자열을 찾아냄"""
    if isinstance(obj, dict):
        for k, v in obj.items():
            if isinstance(k, str) and k.lower() == "answer" and isinstance(v, (str, int, float, bool)):
                return str(v).strip()
        for v in obj.values():
            a = find_answer_recursive(v)
            if a:
                return a
    elif isinstance(obj, list):
        for it in obj:
            a = find_answer_recursive(it)
            if a:
                return a
    return None

def read_answer_any(json_path):
    if not os.path.exists(json_path):
        return None, "path_missing"
    try:
        with open(json_path, "r", encoding="utf-8") as f:
            data = json.load(f)
    except Exception as e:
        return None, f"json_load_error: {e}"
    ans = find_answer_recursive(data)
    if ans is None:
        return None, "answer_key_missing"
    return ans, "ok"

def parse_numeric_dirs(id_dir):
    """subQ 후보: 이름이 숫자(공백/0패딩 허용)이고 1 이상인 디렉터리"""
    raw = [d for d in os.listdir(id_dir) if os.path.isdir(os.path.join(id_dir, d))]
    nums = []
    for name in raw:
        s = name.strip()
        if s.isdigit():
            n = int(s)
            if n >= 1:
                nums.append(n)
    return sorted(set(nums))

def analyze_base_dir(BASE_DIR):
    ids = [d for d in os.listdir(BASE_DIR) if os.path.isdir(os.path.join(BASE_DIR, d))]
    ids.sort()
    total_ids = len(ids)

    with_cot2 = 0
    without_cot2 = 0

    # cot2가 있는 경우에 한해 answer 통계
    ans_no_answer = 0          # "No answer found"
    ans_other = 0              # 그 외 값(문자열/숫자/불리언 등)
    ans_missing_or_error = 0   # a.json 없음, JSON 에러, answer 키 없음

    processed_ids = 0
    for id_name in ids:
        id_dir = os.path.join(BASE_DIR, id_name)
        sub_nums = parse_numeric_dirs(id_dir)
        if not sub_nums:
            # subQ가 없는 아이디는 cot2 없음으로 간주
            without_cot2 += 1
            continue

        last_n = sub_nums[-1]
        cot2_dir = os.path.join(id_dir, str(last_n), "cot2")
        if os.path.isdir(cot2_dir):
            with_cot2 += 1
            ajson = os.path.join(cot2_dir, "a.json")
            ans, st = read_answer_any(ajson)
            if ans is None:
                ans_missing_or_error += 1
            else:
                if ans.strip().lower() == "no answer found":
                    ans_no_answer += 1
                else:
                    ans_other += 1
        else:
            without_cot2 += 1

        processed_ids += 1
        if processed_ids % 50 == 0:
            print(f"[{processed_ids}/{total_ids}] 진행 중...")

    print("\n==== Summary ====")
    print(f"Base dir: {BASE_DIR}")
    print(f"총 아이디 개수: {total_ids}")
    print(f"마지막 subQ 내 cot2 존재: {with_cot2}")
    print(f"마지막 subQ 내 cot2 부재: {without_cot2}")

    print("\n[cot2/a.json의 answer 통계]  (cot2가 있는 아이디만 대상)")
    print(f'  "No answer found": {ans_no_answer}')
    print(f"  그 외(answer 있음): {ans_other}")
    print(f"  a.json 없음/JSON 에러/answer 키 없음: {ans_missing_or_error}")

if __name__ == "__main__":
    # 1) after_cot1
    analyze_base_dir("../data/workspace/after_cot1")
    # 2) Done_CoT2 (필요하면 주석 해제)
    analyze_base_dir("../data/workspace/Done_CoT2")

In [ ]:
import os, json

def find_answer_recursive(obj):
    """dict/list 어디에 있어도 'answer' 키(대소문자 무관)에서 값을 문자열로 찾아냄"""
    if isinstance(obj, dict):
        for k, v in obj.items():
            if isinstance(k, str) and k.lower() == "answer":
                if isinstance(v, (str, int, float, bool)):
                    return str(v).strip()
        for v in obj.values():
            a = find_answer_recursive(v)
            if a is not None:
                return a
    elif isinstance(obj, list):
        for it in obj:
            a = find_answer_recursive(it)
            if a is not None:
                return a
    return None

def read_answer_any(json_path):
    """a.json에서 answer 추출 (없으면 None) + 상태 문자열 반환"""
    if not os.path.exists(json_path):
        return None, "path_missing"
    try:
        with open(json_path, "r", encoding="utf-8") as f:
            data = json.load(f)
    except Exception as e:
        return None, f"json_load_error: {e}"
    ans = find_answer_recursive(data)
    if ans is None:
        return None, "answer_key_missing"
    return ans, "ok"

def parse_numeric_dirs(id_dir):
    """subQ 후보: 이름이 숫자(공백/0패딩 허용) && 1 이상인 디렉터리 → 정수 리스트"""
    raw = [d for d in os.listdir(id_dir) if os.path.isdir(os.path.join(id_dir, d))]
    nums = []
    for name in raw:
        s = name.strip()
        if s.isdigit():
            n = int(s)
            if n >= 1:
                nums.append(n)
    return sorted(set(nums))

def analyze_base_dir(BASE_DIR):
    ids = [d for d in os.listdir(BASE_DIR) if os.path.isdir(os.path.join(BASE_DIR, d))]
    ids.sort()
    total_ids = len(ids)

    # 집계 변수
    considered_ids = 0                 # subQ가 1개 이상인 ID만 카운트
    all_cot2_present_count = 0         # 모든 subQ(1..last)에서 cot2 존재
    any_cot2_missing_count = 0         # subQ 중 하나라도 cot2 미존재
    missing_in_middle_or_last_count = 0
    missing_in_middle_count = 0
    missing_in_last_count = 0
    ids_with_any_no_answer_found = 0   # 어떤 subQ의 cot2/a.json이라도 "No answer found"가 있음

    # 참고용(선택): 스킵된 케이스
    ids_skipped_no_subq = 0

    for id_name in ids:
        id_dir = os.path.join(BASE_DIR, id_name)
        sub_nums = parse_numeric_dirs(id_dir)
        if not sub_nums:
            ids_skipped_no_subq += 1
            continue

        considered_ids += 1
        last_n = sub_nums[-1]

        # subQ별 상태 점검
        cot2_exists_by_n = {}
        no_answer_found_somewhere = False

        for n in sub_nums:
            cot2_dir = os.path.join(id_dir, str(n), "cot2")
            exists = os.path.isdir(cot2_dir)
            cot2_exists_by_n[n] = exists

            if exists:
                ajson = os.path.join(cot2_dir, "a.json")
                ans, st = read_answer_any(ajson)
                if isinstance(ans, str):
                    norm = ans.strip().lower()
                    # "No answer found" 계열 판단(필요시 패턴 추가)
                    if norm == "no answer found" or norm == "no found answer":
                        no_answer_found_somewhere = True

        # 1) 모두 cot2 있는지 / 아닌지
        if all(cot2_exists_by_n.get(n, False) for n in sub_nums):
            all_cot2_present_count += 1
        else:
            any_cot2_missing_count += 1

        # 2) 중간(2..last-1) 또는 마지막(last)에 없는 경우 집계
        #    - 중간: sub_nums에 2..last-1이 존재하고 그중 하나라도 cot2 없음
        #    - 마지막: last_n에 cot2 없음
        missing_middle = False
        if len(sub_nums) >= 3:
            for n in sub_nums[1:-1]:  # 2..last-1
                if not cot2_exists_by_n.get(n, False):
                    missing_middle = True
                    break
        missing_last = not cot2_exists_by_n.get(last_n, False)

        if missing_middle:
            missing_in_middle_count += 1
        if missing_last:
            missing_in_last_count += 1
        if missing_middle or missing_last:
            missing_in_middle_or_last_count += 1

        # 3) 어떤 cot2라도 "No answer found"가 있는 ID
        if no_answer_found_somewhere:
            ids_with_any_no_answer_found += 1

    # 출력
    print("\n==== Summary ====")
    print(f"Base dir: {BASE_DIR}")
    print(f"총 아이디 수(디렉터리): {total_ids}")
    print(f"subQ가 1개 이상인 아이디 수: {considered_ids}")
    print(f"subQ가 하나도 없는 아이디(스킵): {ids_skipped_no_subq}")

    print("\n[cot2 존재 여부]")
    print(f"  마지막 subQ까지 '모든' subQ에 cot2가 있는 ID 수: {all_cot2_present_count}")
    print(f"  (보완) subQ 중 하나라도 cot2가 없는 ID 수: {any_cot2_missing_count}")

    print("\n[cot2 결손 위치]")
    print(f"  중간 또는 마지막에 cot2가 없는 ID 수(통합): {missing_in_middle_or_last_count}")
    print(f"    ├─ 중간 subQ(2..last-1)에 없는 ID 수: {missing_in_middle_count}")
    print(f"    └─ 마지막 subQ(last)에 없는 ID 수: {missing_in_last_count}")

    print("\n[answer가 'No answer found'인 경우]")
    print(f"  어떤 subQ의 cot2에도 'No answer found'가 하나라도 있는 ID 수: {ids_with_any_no_answer_found}")

if __name__ == "__main__":
    # 1) after_cot1
    #analyze_base_dir("../data/workspace/after_cot1")
    # 2) Done_CoT2
    analyze_base_dir("../data/workspace/Done_CoT2")

In [ ]:
import os, json, random, re, string
from typing import Dict, List, Tuple

BASE_DIR = "../data/workspace/Done_CoT2"
GOLD_PATH = "../data/workspace/hotpot_qas_val.json"
SAMPLE_K = 50
RANDOM_SEED = 42

# ---------- 유틸: 정답 추출 ----------
def find_answer_recursive(obj):
    """dict/list 어디에 있어도 'answer' 키(대소문자 무관)에서 값을 문자열로 찾아냄"""
    if isinstance(obj, dict):
        for k, v in obj.items():
            if isinstance(k, str) and k.lower() == "answer":
                # 값이 리스트면 첫 원소 기준, 아니면 문자열화
                if isinstance(v, list) and v:
                    return str(v[0]).strip()
                if isinstance(v, (str, int, float, bool)):
                    return str(v).strip()
        for v in obj.values():
            a = find_answer_recursive(v)
            if a is not None:
                return a
    elif isinstance(obj, list):
        for it in obj:
            a = find_answer_recursive(it)
            if a is not None:
                return a
    return None

def read_pred_answer_0ajson(id_dir: str):
    """ID/0/a.json 에서 answer 추출"""
    path = os.path.join(id_dir, "0", "a.json")
    if not os.path.exists(path):
        return None
    try:
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
    except Exception:
        return None
    ans = find_answer_recursive(data)
    return ans

# ---------- 유틸: gold 로더 (JSON 또는 JSONL 모두 지원) ----------
def load_hotpot_gold(path: str) -> Dict[str, List[str]]:
    """
    반환: {_id: [정답1, 정답2, ...]} (정답이 문자열이면 리스트로 감쌈)
    """
    items = None
    try:
        with open(path, "r", encoding="utf-8") as f:
            items = json.load(f)      # JSON 리스트 형식 시도
        if not isinstance(items, list):
            raise ValueError("Not a JSON list")
    except Exception:
        # JSONL로 가정하고 라인별 파싱
        items = []
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    items.append(json.loads(line))
                except Exception:
                    pass

    id2gold: Dict[str, List[str]] = {}
    for ex in items:
        _id = ex.get("_id")
        if not _id:
            continue
        gold = ex.get("answer")
        if gold is None:
            continue
        if isinstance(gold, list):
            golds = [str(x).strip() for x in gold if str(x).strip()]
        else:
            golds = [str(gold).strip()]
        if golds:
            id2gold[_id] = golds
    return id2gold

# ---------- 평가: normalize / EM / F1 ----------
def normalize_answer(s: str) -> str:
    """SQuAD 스타일 정규화 (소문자, 기사/문장부호 제거, 공백 정리)"""
    def lower(text): return text.lower()
    def remove_punc(text):
        return "".join(ch for ch in text if ch not in set(string.punctuation))
    def remove_articles(text):
        return re.sub(r"\b(a|an|the)\b", " ", text)
    def white_space_fix(text):
        return " ".join(text.split())
    return white_space_fix(remove_articles(remove_punc(lower(s))))

def f1_score(pred: str, gold: str) -> float:
    pred_tokens = normalize_answer(pred).split()
    gold_tokens = normalize_answer(gold).split()
    if len(pred_tokens) == 0 and len(gold_tokens) == 0:
        return 1.0
    if len(pred_tokens) == 0 or len(gold_tokens) == 0:
        return 0.0
    common = {}
    for t in pred_tokens:
        common[t] = common.get(t, 0) + 1
    num_same = 0
    for t in gold_tokens:
        if common.get(t, 0) > 0:
            num_same += 1
            common[t] -= 1
    if num_same == 0:
        return 0.0
    precision = num_same / len(pred_tokens)
    recall = num_same / len(gold_tokens)
    return 2 * precision * recall / (precision + recall)

def exact_match_score(pred: str, gold: str) -> int:
    return int(normalize_answer(pred) == normalize_answer(gold))

def max_over_golds(pred: str, golds: List[str]) -> Tuple[int, float]:
    em = max(exact_match_score(pred, g) for g in golds)
    f1 = max(f1_score(pred, g) for g in golds)
    return em, f1

# ---------- 메인 ----------
def main():
    # 예측(=Done_CoT2/ID/0/a.json) 수집
    id_dirs = [d for d in os.listdir(BASE_DIR) if os.path.isdir(os.path.join(BASE_DIR, d))]
    id_dirs.sort()
    id2pred: Dict[str, str] = {}
    for _id in id_dirs:
        ans = read_pred_answer_0ajson(os.path.join(BASE_DIR, _id))
        if ans is not None:
            id2pred[_id] = ans

    # 골드 로드
    id2gold = load_hotpot_gold(GOLD_PATH)

    # 교집합으로 평가
    common_ids = sorted(set(id2pred.keys()) & set(id2gold.keys()))
    n = len(common_ids)
    if n == 0:
        print("교집합 ID가 없습니다. 경로/파일을 확인하세요.")
        return

    total_em = 0
    total_f1 = 0.0
    per_example = []
    for _id in common_ids:
        pred = id2pred[_id]
        golds = id2gold[_id]
        em, f1 = max_over_golds(pred, golds)
        total_em += em
        total_f1 += f1
        per_example.append({
            "_id": _id,
            "pred": pred,
            "golds": golds,
            "em": em,
            "f1": f1
        })

    em_score = 100.0 * total_em / n
    f1_score_avg = 100.0 * total_f1 / n

    # 랜덤 샘플 50개 출력
    random.seed(RANDOM_SEED)
    k = min(SAMPLE_K, n)
    sample = random.sample(per_example, k)
    print(f"\n=== Random {k} / {n} samples ===")
    for i, ex in enumerate(sample, 1):
        print(f"[{i}] ID={ex['_id']}")
        print(f"    pred = {ex['pred']}")
        print(f"    golds= {ex['golds']}")
        print(f"    EM={ex['em']}  F1={ex['f1']:.2f}")

    print("\n=== Overall ===")
    print(f"Evaluated examples: {n}")
    print(f"EM = {em_score:.2f}")
    print(f"F1 = {f1_score_avg:.2f}")

if __name__ == "__main__":
    main()

In [ ]:
import os, json, random, re, string
from typing import Dict, List, Tuple

BASE_DIR = "../data/workspace/Done_CoT2"
GOLD_PATH = "../data/workspace/hotpot_qas_val.json"
SAMPLE_K = 50
RANDOM_SEED = 42

# ---------- metrics ----------
def find_answer_recursive(obj):
    # 'answer' 키를 우선적으로 깊게 파고들어 문자열을 찾아오는 버전
    if isinstance(obj, dict):
        # 1) 먼저 'answer' 키들을 우선 처리
        for k, v in obj.items():
            if isinstance(k, str) and k.lower() == "answer":
                # 문자열/숫자/불리언
                if isinstance(v, (str, int, float, bool)):
                    s = str(v).strip()
                    if s:
                        return s
                # 리스트: 내부에서 문자열이나 dict(answer) 탐색
                if isinstance(v, list):
                    for it in v:
                        s = find_answer_recursive(it)
                        if s:
                            return s
                # dict: 특히 0/a.json처럼 {"answer": {"answer": "..."}} 케이스
                if isinstance(v, dict):
                    # 1) 바로 내부 'answer' 키 시도
                    inner = v.get("answer", None)
                    if isinstance(inner, (str, int, float, bool)):
                        s = str(inner).strip()
                        if s:
                            return s
                    # 2) 못 찾으면 재귀적으로 더 내려감
                    s = find_answer_recursive(v)
                    if s:
                        return s
        # 2) 그 외 키들도 재귀 탐색
        for v in obj.values():
            s = find_answer_recursive(v)
            if s:
                return s
    elif isinstance(obj, list):
        for it in obj:
            s = find_answer_recursive(it)
            if s:
                return s
    return None
def normalize_answer(s: str) -> str:
    def lower(t): return t.lower()
    def remove_punc(t): return "".join(ch for ch in t if ch not in set(string.punctuation))
    def remove_articles(t): return re.sub(r"\b(a|an|the)\b", " ", t)
    def ws(t): return " ".join(t.split())
    return ws(remove_articles(remove_punc(lower(s))))

def f1_score(pred: str, gold: str) -> float:
    pt = normalize_answer(pred).split()
    gt = normalize_answer(gold).split()
    if not pt and not gt: return 1.0
    if not pt or not gt: return 0.0
    common = {}
    for t in pt: common[t] = common.get(t, 0) + 1
    same = 0
    for t in gt:
        if common.get(t, 0) > 0:
            same += 1
            common[t] -= 1
    if same == 0: return 0.0
    prec = same / len(pt)
    rec  = same / len(gt)
    return 2 * prec * rec / (prec + rec)

def exact_match_score(pred: str, gold: str) -> int:
    return int(normalize_answer(pred) == normalize_answer(gold))

def max_over_golds(pred: str, golds: List[str]) -> Tuple[int, float]:
    return max(exact_match_score(pred, g) for g in golds), max(f1_score(pred, g) for g in golds)

# ---------- io utils ----------
def read_json(path: str):
    if not os.path.exists(path): return None
    try:
        with open(path, "r", encoding="utf-8") as f: return json.load(f)
    except Exception: return None

def find_value_recursive(obj, key_lower: str):
    if isinstance(obj, dict):
        for k, v in obj.items():
            if isinstance(k, str) and k.lower() == key_lower:
                return v
        for v in obj.values():
            r = find_value_recursive(v, key_lower)
            if r is not None: return r
    elif isinstance(obj, list):
        for it in obj:
            r = find_value_recursive(it, key_lower)
            if r is not None: return r
    return None

def find_answer_recursive(obj) -> str | None:
    v = find_value_recursive(obj, "answer")
    if v is None: return None
    if isinstance(v, list): return str(v[0]).strip() if v else None
    if isinstance(v, (str, int, float, bool)): return str(v).strip()
    return None

def load_hotpot_gold(path: str) -> Dict[str, List[str]]:
    # JSON(list) or JSONL
    try:
        with open(path, "r", encoding="utf-8") as f:
            items = json.load(f)
        if not isinstance(items, list): raise ValueError
    except Exception:
        items = []
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    try: items.append(json.loads(line))
                    except Exception: pass
    id2gold = {}
    for ex in items:
        _id = ex.get("_id"); gold = ex.get("answer")
        if not _id or gold is None: continue
        if isinstance(gold, list): golds = [str(x).strip() for x in gold if str(x).strip()]
        else: golds = [str(gold).strip()]
        if golds: id2gold[_id] = golds
    return id2gold

def parse_numeric_dirs(id_dir: str) -> List[int]:
    if not os.path.isdir(id_dir): return []
    raw = [d for d in os.listdir(id_dir) if os.path.isdir(os.path.join(id_dir, d))]
    nums = []
    for name in raw:
        s = name.strip()
        if s.isdigit():
            n = int(s)
            if n >= 1: nums.append(n)
    return sorted(set(nums))

def is_negative_no_answer(ans: str) -> bool:
    a = normalize_answer(ans)
    return a in {"no answer found", "no found answer", "no answer"}

# ---------- build predictions from BASE_DIR ids ----------
def get_base_dir_ids(base_dir: str) -> List[str]:
    ids = [d for d in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, d))]
    ids.sort()
    return ids

def get_final_predictions_from_basedir(base_dir: str, ids: List[str]):
    """
    각 ID에 대해:
      1) 마지막 subQ의 cot2/a.json answer가 유효하면 -> 사용
      2) 아니면 0/a.json answer로 대체
    return: (id2pred, used_cot2, used_base, missing)
    """
    id2pred = {}
    used_cot2 = used_base = missing = 0

    for _id in ids:
        id_dir = os.path.join(base_dir, _id)
        sub_nums = parse_numeric_dirs(id_dir)

        used = False
        # 1) last cot2
        if sub_nums:
            last_n = sub_nums[-1]
            ajson = os.path.join(id_dir, str(last_n), "cot2", "a.json")
            data_c = read_json(ajson)
            if data_c is not None:
                ans_c = find_answer_recursive(data_c)
                if ans_c is not None and not is_negative_no_answer(ans_c):
                    id2pred[_id] = ans_c
                    used_cot2 += 1
                    used = True

        # 2) fallback: 0/a.json
        if not used:
            bjson = os.path.join(id_dir, "0", "a.json")
            data_b = read_json(bjson)
            if data_b is not None:
                ans_b = find_answer_recursive(data_b)
                if ans_b is not None:
                    id2pred[_id] = ans_b
                    used_base += 1
                    used = True

        if not used:
            missing += 1

    return id2pred, used_cot2, used_base, missing


# ---------- evaluate over BASE_DIR ids (gold가 있는 것만 점수 계산) ----------
def evaluate_on_ids(id2pred: Dict[str, str], id2gold: Dict[str, List[str]], ids: List[str]):
    eval_ids = [i for i in ids if i in id2gold]   # gold이 있는 id만 평가
    missing_gold = len(ids) - len(eval_ids)

    total_em = 0
    total_f1 = 0.0
    rows = []
    for _id in eval_ids:
        pred = id2pred.get(_id)
        golds = id2gold[_id]
        if pred is None:
            em, f1 = 0, 0.0
        else:
            em, f1 = max_over_golds(pred, golds)
        total_em += em
        total_f1 += f1
        rows.append({"_id": _id, "pred": pred, "golds": golds, "em": em, "f1": f1})

    N = len(eval_ids)
    em_pct = 100.0 * total_em / N if N else 0.0
    f1_pct = 100.0 * total_f1 / N if N else 0.0
    return N, em_pct, f1_pct, rows, missing_gold

def print_samples(rows: List[dict], k: int):
    rows_with_pred = [r for r in rows if r["pred"] is not None]
    random.seed(RANDOM_SEED)
    sample = random.sample(rows_with_pred, min(k, len(rows_with_pred)))
    print(f"\n=== Random {len(sample)} / {len(rows)} samples (with pred) ===")
    for i, ex in enumerate(sample, 1):
        print(f"[{i}] ID={ex['_id']}")
        print(f"    pred = {ex['pred']}")
        print(f"    golds= {ex['golds']}")
        print(f"    EM={ex['em']}  F1={ex['f1']:.2f}")

def main():
    # BASE_DIR의 ID(=505 기대)
    base_ids = get_base_dir_ids(BASE_DIR)
    print(f"BASE_DIR ids: {len(base_ids)}")

    # 최종 예측 생성 (cot2 우선, 나머지 0/a.json)
    id2pred, used_cot2, used_base, missing = get_final_predictions_from_basedir(BASE_DIR, base_ids)
    print("\n=== Fill counts (from BASE_DIR) ===")
    print(f"From last cot2: {used_cot2}")
    print(f"From 0/a.json : {used_base}")
    print(f"Missing       : {missing}  (파일/answer 누락)")

    # gold 로드
    id2gold = load_hotpot_gold(GOLD_PATH)

    # 평가: BASE_DIR의 id들 중 gold가 있는 경우만 분모로 (일반적인 평가 관례)
    N, em, f1, rows, missing_gold = evaluate_on_ids(id2pred, id2gold, base_ids)
    print("\n=== Scores over BASE_DIR ids that have gold ===")
    print(f"N (evaluated) = {N}  (BASE_DIR {len(base_ids)}, gold missing for {missing_gold})")
    print(f"EM = {em:.2f}")
    print(f"F1 = {f1:.2f}")

    # 샘플 50개
    print_samples(rows, SAMPLE_K)

if __name__ == "__main__":
    main()